# SheCare INT 234: Data Preprocessing and Cleaning

This notebook performs preprocessing and cleaning only. It uses the actual EDA
findings from the previous notebook and does not train a machine-learning model.
The original CSV is loaded read-only and a separate cleaned CSV is created.

Important principles:
- Blank and whitespace-only cells become missing values in a working copy.
- Only exact duplicate rows are removed.
- Repeated participant records are retained because they can be genuine cycles.
- Missing predictors are not filled using the full dataset.
- `ClientID` is retained for participant-aware validation but is excluded from
  predictive input features.
- `LengthofCycle` is investigated as a possible regression target, not assumed
  to be a final target without considering prediction timing and leakage.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

project_dir = Path.cwd()
if (project_dir / "dataset" / "raw").exists():
    shecare_dir = project_dir
elif (project_dir.parent / "dataset" / "raw").exists():
    shecare_dir = project_dir.parent
else:
    shecare_dir = project_dir / "SheCare_INT234"

raw_file = shecare_dir / "dataset" / "raw" / "FedCycleData071012.csv"
processed_dir = shecare_dir / "dataset" / "processed"
result_dir = shecare_dir / "results"
processed_dir.mkdir(parents=True, exist_ok=True)
result_dir.mkdir(parents=True, exist_ok=True)
print("Project directory:", shecare_dir)

Project directory: /Users/abhijeet.ist/Developer/SheSync/SheCare_INT234


## 1. Load the original data and create a working copy

In [2]:
original_df = pd.read_csv(raw_file, keep_default_na=False)
cleaned_df = original_df.copy()

before_rows = len(cleaned_df)
before_columns = len(cleaned_df.columns)
print("Rows before cleaning:", before_rows)
print("Columns before cleaning:", before_columns)
print("Original data remains stored in original_df and is not modified.")

Rows before cleaning: 1665
Columns before cleaning: 80
Original data remains stored in original_df and is not modified.


The EDA found 1,665 rows and 80 columns. It also found that many missing values
are stored as a single space or an empty string. These values are standardized
only in `cleaned_df`.

In [3]:
blank_counts = {}
for column in cleaned_df.columns:
    blank_counts[column] = cleaned_df[column].astype(str).str.strip().eq("").sum()

blank_counts = pd.Series(blank_counts).sort_values(ascending=False)
print("Blank or whitespace-only cells before standardization:", int(blank_counts.sum()))

for column in cleaned_df.columns:
    cleaned_df[column] = cleaned_df[column].replace(r"^\s*$", np.nan, regex=True)

print("Missing cells after standardization:", int(cleaned_df.isna().sum().sum()))

Blank or whitespace-only cells before standardization: 88164
Missing cells after standardization: 88164


## 2. Handle duplicates and repeated participant-cycle records

In [4]:
exact_duplicate_count = int(cleaned_df.duplicated().sum())
repeated_key_mask = cleaned_df.duplicated(["ClientID", "CycleNumber"], keep=False)
repeated_key_rows = cleaned_df.loc[
    repeated_key_mask, ["ClientID", "CycleNumber", "LengthofCycle"]
].copy()

print("Exact duplicate rows:", exact_duplicate_count)
print("Rows in repeated ClientID and CycleNumber groups:", len(repeated_key_rows))
print("Repeated keys are retained because they are not all identical.")
display(repeated_key_rows.head(20))

Exact duplicate rows: 12
Rows in repeated ClientID and CycleNumber groups: 32
Repeated keys are retained because they are not all identical.


,ClientID,CycleNumber,LengthofCycle
45,nfp8114,1,28
46,nfp8114,2,34
47,nfp8109,1,29
48,nfp8109,2,23
49,nfp8109,3,27
50,nfp8107,1,28
51,nfp8107,2,27
52,nfp8107,3,25
53,nfp8107,4,25
54,nfp8107,5,29


In [5]:
duplicate_rows = cleaned_df[cleaned_df.duplicated(keep=False)].copy()
if len(duplicate_rows) > 0:
    display(duplicate_rows.head())

# Remove only exact duplicate rows and keep the first copy.
cleaned_df = cleaned_df.drop_duplicates(keep="first").reset_index(drop=True)
after_duplicate_rows = len(cleaned_df)
print("Rows after removing exact duplicates:", after_duplicate_rows)
print("Rows removed:", before_rows - after_duplicate_rows)
print("Columns after duplicate step:", len(cleaned_df.columns))

,ClientID,CycleNumber,Group,CycleWithPeakorNot,ReproductiveCategory,LengthofCycle,MeanCycleLength,EstimatedDayofOvulation,LengthofLutealPhase,FirstDayofHigh,...,Method,Prevmethod,Methoddate,Whychart,Nextpreg,NextpregM,Spousesame,SpousesameM,Timeattemptpreg,BMI
45,nfp8114,1,0,1,0,28,31,14,14,NaN,...,9,NaN,NaN,2,3,3,1,1,0,28.1260813148789
47,nfp8109,1,0,1,0,29,26.33,19,10,14,...,9,NaN,NaN,2,3,3,1,1,0,29.0495867768595
48,nfp8109,2,0,1,0,23,NaN,14,9,9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49,nfp8109,3,0,1,0,27,NaN,17,10,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
52,nfp8107,3,0,1,0,25,NaN,16,9,10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Rows after removing exact duplicates: 1653
Rows removed: 12
Columns after duplicate step: 80


The 12 exact duplicates were removed. The repeated `ClientID` and `CycleNumber`
combinations were not removed because the EDA showed that some repeated keys
have different cycle values. Participant-level grouping should be used later
instead of deleting records arbitrarily.

## 3. Convert clearly numeric columns

In [6]:
numeric_columns = [
    "CycleNumber", "Group", "CycleWithPeakorNot", "ReproductiveCategory",
    "LengthofCycle", "MeanCycleLength", "EstimatedDayofOvulation",
    "LengthofLutealPhase", "FirstDayofHigh", "TotalNumberofHighDays",
    "TotalHighPostPeak", "TotalNumberofPeakDays", "TotalDaysofFertility",
    "TotalFertilityFormula", "LengthofMenses", "MeanMensesLength",
    "MensesScoreDayOne", "MensesScoreDayTwo", "MensesScoreDayThree",
    "MensesScoreDayFour", "MensesScoreDayFive", "MensesScoreDaySix",
    "MensesScoreDaySeven", "MensesScoreDayEight", "MensesScoreDayNine",
    "MensesScoreDayTen", "MensesScoreDay11", "MensesScoreDay12",
    "MensesScoreDay13", "MensesScoreDay14", "MensesScoreDay15",
    "TotalMensesScore", "MeanBleedingIntensity", "NumberofDaysofIntercourse",
    "IntercourseInFertileWindow", "UnusualBleeding", "PhasesBleeding",
    "IntercourseDuringUnusBleed", "Age", "AgeM", "Yearsmarried", "Wedding",
    "Schoolyears", "SchoolyearsM", "IncomeM", "Height", "Weight",
    "Numberpreg", "Livingkids", "Miscarriages", "Abortions", "Medvits",
    "Gynosurgeries", "LivingkidsM", "Boys", "Girls", "MedvitsM",
    "Urosurgeries", "Breastfeeding", "Method", "Prevmethod", "Nextpreg",
    "NextpregM", "Spousesame", "SpousesameM", "Timeattemptpreg", "BMI"
]
numeric_columns = [column for column in numeric_columns if column in cleaned_df.columns]

conversion_summary = []
for column in numeric_columns:
    old_non_missing = cleaned_df[column].notna().sum()
    converted = pd.to_numeric(cleaned_df[column], errors="coerce")
    new_non_missing = converted.notna().sum()
    conversion_summary.append({
        "column": column,
        "non_missing_before": old_non_missing,
        "non_missing_after": new_non_missing,
        "values_changed_to_missing": old_non_missing - new_non_missing
    })
    cleaned_df[column] = converted

conversion_summary = pd.DataFrame(conversion_summary)
conversion_summary.to_csv(result_dir / "numeric_conversion_summary.csv", index=False)
print("Numeric columns converted:", len(numeric_columns))
display(conversion_summary[conversion_summary["values_changed_to_missing"] > 0])

Numeric columns converted: 67


,column,non_missing_before,non_missing_after,values_changed_to_missing
52,Gynosurgeries,26,0,26
57,Urosurgeries,3,0,3


Numeric-looking columns are converted with `errors="coerce"` so invalid text
becomes missing instead of silently remaining mixed-type. Identifier and
explanatory text columns are not forced to numeric. The conversion summary
records any values that became missing during conversion.

## 4. Check invalid and suspicious values

In [7]:
range_checks = []
checks = {
    "LengthofCycle": (1, 100),
    "LengthofMenses": (1, 31),
    "Age": (12, 100),
    "BMI": (5, 100),
    "Height": (20, 100),
    "Weight": (20, 500)
}

for column, limits in checks.items():
    if column in cleaned_df.columns:
        values = cleaned_df[column].dropna()
        low, high = limits
        suspicious = ((values < low) | (values > high)).sum()
        range_checks.append({
            "column": column,
            "lower_review_limit": low,
            "upper_review_limit": high,
            "suspicious_values": int(suspicious),
            "valid_values": int(values.notna().sum())
        })

range_checks = pd.DataFrame(range_checks)
range_checks.to_csv(result_dir / "range_check_summary.csv", index=False)
display(range_checks)

,column,lower_review_limit,upper_review_limit,suspicious_values,valid_values
0,LengthofCycle,1,100,0,1653
1,LengthofMenses,1,31,0,1649
2,Age,12,100,0,139
3,BMI,5,100,0,128
4,Height,20,100,0,130
5,Weight,20,500,1,131


In [8]:
categorical_columns = [
    "ClientID", "Maristatus", "MaristatusM", "Religion", "ReligionM",
    "Ethnicity", "EthnicityM", "OccupationM", "Reprocate", "Medvitexplain",
    "MedvitexplainM", "Methoddate", "Whychart"
]
label_summary = []
for column in categorical_columns:
    if column in cleaned_df.columns:
        values = cleaned_df[column].dropna().astype(str)
        label_summary.append({
            "column": column,
            "valid_values": len(values),
            "unique_values": values.nunique(),
            "values_with_outer_whitespace": int((values != values.str.strip()).sum())
        })
label_summary = pd.DataFrame(label_summary)
label_summary.to_csv(result_dir / "categorical_label_summary.csv", index=False)
display(label_summary)

,column,valid_values,unique_values,values_with_outer_whitespace
0,ClientID,1653,159,0
1,Maristatus,139,3,0
2,MaristatusM,139,4,0
3,Religion,139,3,0
4,ReligionM,139,3,0
5,Ethnicity,138,5,0
6,EthnicityM,139,5,0
7,OccupationM,138,7,0
8,Reprocate,139,7,0
9,Medvitexplain,103,78,0


Range checks are review checks, not automatic deletion rules. The unit or
definition of some source fields is not fully available, so suspicious values
are retained for source verification. Whitespace was already removed when
blank cells were standardized; category spelling and code meanings should be
confirmed with the dataset documentation.

## 5. Missing-value decisions

In [9]:
missing_summary = pd.DataFrame({
    "missing_count": cleaned_df.isna().sum(),
    "missing_percent": (cleaned_df.isna().mean() * 100).round(2),
    "valid_count": cleaned_df.notna().sum()
}).sort_values("missing_percent", ascending=False)
missing_summary.to_csv(result_dir / "preprocessing_missing_summary.csv")
display(missing_summary.head(20))

,missing_count,missing_percent,valid_count
Gynosurgeries,1653,100.00,0
Urosurgeries,1653,100.00,0
MensesScoreDay13,1652,99.94,1
MensesScoreDay15,1652,99.94,1
MensesScoreDay14,1652,99.94,1
MensesScoreDay12,1652,99.94,1
MensesScoreDay11,1651,99.88,2
MensesScoreDayTen,1647,99.64,6
Wedding,1643,99.40,10
MensesScoreDayNine,1627,98.43,26


Missing-value decisions for this phase:

- No columns are removed only because they have high missingness. Their
  usefulness depends on the later prediction design and their definitions.
- No predictor values are imputed in the exported dataset. Imputation should
  be fitted only on the training split in the modelling phase.
- `LengthofCycle` is the candidate target. Its missing rows must be excluded
  from supervised training rather than target-imputed. In this supplied file,
  the target has no missing values.
- Text fields with missing values remain missing because filling them with a
  made-up category could change their meaning.

In [10]:
target_column = "LengthofCycle"
target_missing_count = int(cleaned_df[target_column].isna().sum())
print("Missing candidate target values:", target_missing_count)
print("Rows available with a valid candidate target:", int(cleaned_df[target_column].notna().sum()))

Missing candidate target values: 0
Rows available with a valid candidate target: 1653


## 6. Outlier review

In [11]:
outlier_columns = [
    "LengthofCycle", "LengthofMenses", "EstimatedDayofOvulation",
    "LengthofLutealPhase", "TotalDaysofFertility", "TotalMensesScore",
    "Age", "BMI", "Height", "Weight"
]
outlier_rows = []
for column in outlier_columns:
    if column in cleaned_df.columns:
        values = cleaned_df[column].dropna()
        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        count = ((values < lower) | (values > upper)).sum()
        outlier_rows.append({
            "column": column,
            "lower_boundary": lower,
            "upper_boundary": upper,
            "potential_outliers": int(count)
        })
outlier_summary = pd.DataFrame(outlier_rows)
outlier_summary.to_csv(result_dir / "preprocessing_outlier_review.csv", index=False)
display(outlier_summary.round(2))

,column,lower_boundary,upper_boundary,potential_outliers
0,LengthofCycle,21.00,37.00,73
1,LengthofMenses,1.00,9.00,6
2,EstimatedDayofOvulation,8.00,24.00,44
3,LengthofLutealPhase,9.00,17.00,97
4,TotalDaysofFertility,0.00,16.00,38
5,TotalMensesScore,3.50,15.50,72
6,Age,12.50,48.50,0
7,BMI,11.18,37.39,5
8,Height,57.00,73.00,0
9,Weight,60.00,236.00,5


Potential outliers are retained. The EDA showed that an IQR flag is only a
screening result and does not prove that a measurement is wrong. Later work
can compare flagged values with the source record and domain documentation.

## 7. Candidate target and feature decisions

In [12]:
retained_for_cleaned_file = cleaned_df.columns.tolist()

excluded_from_predictors = {
    "ClientID": "Participant identifier; retain separately for participant-aware validation.",
    "LengthofCycle": "Candidate target, not an input feature.",
    "MeanCycleLength": "May summarize cycle length and can leak the target.",
    "EstimatedDayofOvulation": "May be calculated from the cycle and may not be known at prediction time.",
    "LengthofLutealPhase": "Cycle-derived measurement; possible target leakage.",
    "TotalDaysofFertility": "Cycle-derived measurement; possible target leakage.",
    "TotalFertilityFormula": "Formula-derived fertility field; possible target leakage.",
    "FirstDayofHigh": "Cycle-derived timing field; possible target leakage.",
    "TotalNumberofHighDays": "Cycle-derived field; possible target leakage.",
    "TotalHighPostPeak": "Cycle-derived field; possible target leakage.",
    "TotalNumberofPeakDays": "Cycle-derived field; possible target leakage.",
    "CycleWithPeakorNot": "May summarize information identified during the cycle; timing must be checked.",
    "LengthofMenses": "May be known after the cycle and must be checked against the intended prediction time.",
    "TotalMensesScore": "May be known after the cycle and must be checked against the intended prediction time."
}
excluded_from_predictors = {
    column: reason for column, reason in excluded_from_predictors.items()
    if column in cleaned_df.columns
}
retained_candidate_predictors = [
    column for column in cleaned_df.columns
    if column not in excluded_from_predictors
]
feature_decisions = []
for column in retained_candidate_predictors:
    feature_decisions.append({
        "column": column,
        "decision": "Retained in cleaned dataset; candidate predictor only after timing and missingness review.",
        "reason": "Not identified as an identifier, target, or directly cycle-derived field in this phase."
    })
for column, reason in excluded_from_predictors.items():
    feature_decisions.append({
        "column": column,
        "decision": "Excluded from candidate predictor list.",
        "reason": reason
    })
feature_decisions = pd.DataFrame(feature_decisions).sort_values("column")
feature_decisions.to_csv(result_dir / "retained_and_excluded_features.csv", index=False)
print("Candidate target:", target_column)
print("Retained candidate predictor count:", len(retained_candidate_predictors))
print("Excluded from candidate predictors:", len(excluded_from_predictors))
display(feature_decisions)

Candidate target: LengthofCycle
Retained candidate predictor count: 66
Excluded from candidate predictors: 14


,column,decision,reason
45,Abortions,Retained in cleaned dataset; candidate predict...,"Not identified as an identifier, target, or di..."
25,Age,Retained in cleaned dataset; candidate predict...,"Not identified as an identifier, target, or di..."
26,AgeM,Retained in cleaned dataset; candidate predict...,"Not identified as an identifier, target, or di..."
65,BMI,Retained in cleaned dataset; candidate predict...,"Not identified as an identifier, target, or di..."
50,Boys,Retained in cleaned dataset; candidate predict...,"Not identified as an identifier, target, or di..."
...,...,...,...
54,Urosurgeries,Retained in cleaned dataset; candidate predict...,"Not identified as an identifier, target, or di..."
30,Wedding,Retained in cleaned dataset; candidate predict...,"Not identified as an identifier, target, or di..."
40,Weight,Retained in cleaned dataset; candidate predict...,"Not identified as an identifier, target, or di..."
59,Whychart,Retained in cleaned dataset; candidate predict...,"Not identified as an identifier, target, or di..."


The cleaned CSV retains all cleaned columns so that no potentially useful
information is irreversibly discarded during preprocessing. The feature
decision table separately identifies columns that should not be used as
predictors without a clear prediction-time definition. `ClientID` remains
available for participant-aware splitting, but it is not a predictive input.

## 8. Export cleaned data and cleaning summary

In [13]:
cleaned_file = processed_dir / "cleaned_dataset.csv"
cleaned_df.to_csv(cleaned_file, index=False)

cleaning_summary = pd.DataFrame([
    {
        "issue_identified": "Blank and whitespace-only cells",
        "action_taken": "Converted to missing values in the working copy",
        "reason": "The EDA showed blank strings used as missing markers; this preserves missingness without inventing values."
    },
    {
        "issue_identified": "12 exact duplicate rows",
        "action_taken": "Removed exact duplicates and kept the first copy",
        "reason": "Exact duplicates do not add a new observation."
    },
    {
        "issue_identified": "Repeated ClientID and CycleNumber combinations",
        "action_taken": "Retained repeated records",
        "reason": "The repeated-key records were not all identical; they may represent real observations and need source review."
    },
    {
        "issue_identified": "Mixed numeric and text storage",
        "action_taken": "Converted defined numeric columns with invalid text becoming missing",
        "reason": "Numeric analysis and later feature preparation need consistent numeric types."
    },
    {
        "issue_identified": "Missing predictor values",
        "action_taken": "Retained missing values; no full-dataset imputation",
        "reason": "Imputers must be fitted on training data only during modelling."
    },
    {
        "issue_identified": "Missing candidate target values",
        "action_taken": "No target imputation",
        "reason": "Rows without a target cannot be used for supervised regression; this dataset currently has zero missing target values."
    },
    {
        "issue_identified": "Potential outliers",
        "action_taken": "Flagged for review and retained",
        "reason": "An IQR flag does not prove a data-entry error."
    },
    {
        "issue_identified": "Identifiers and possible leakage fields",
        "action_taken": "Retained in cleaned CSV but excluded from candidate predictor list",
        "reason": "ClientID is an identifier and several fields may be calculated after or from the target."
    }
])
cleaning_summary.to_csv(result_dir / "cleaning_summary.csv", index=False)

print("Rows after cleaning:", len(cleaned_df))
print("Columns after cleaning:", len(cleaned_df.columns))
print("Cleaned file:", cleaned_file)
print("Raw file was not overwritten.")

Rows after cleaning: 1653
Columns after cleaning: 80
Cleaned file: /Users/abhijeet.ist/Developer/SheSync/SheCare_INT234/dataset/processed/cleaned_dataset.csv
Raw file was not overwritten.


## Remaining limitations

The cleaned data still contains missing values because this phase avoids
leakage-prone full-dataset imputation. The meanings and units of coded fields
such as `Group`, `Method` and reproductive categories should be confirmed with
the data dictionary. Repeated participant records require participant-aware
train-test splitting later. Finally, the prediction-time availability of
cycle-derived variables must be confirmed before selecting modelling features.

No machine-learning model was trained in this notebook.